# XGBoost Regression Model with BLOSUM62 + Physicochemical features
Uses 5-Fold Position-Split, trains XGBoost ensemble

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [2]:
!pip install xgboost biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.9 MB/s eta 0:00:00


In [3]:
import os
import xgboost as xgb
from sklearn.model_selection import KFold
from Bio.Align import substitution_matrices

## 1. Load Data

In [8]:
TRAIN_PATH = 'train.csv'
TEST_PATH  = 'test.csv'
FASTA_PATH = 'sequence.fasta'
QUERY_PATHS = ['query1_labeled.csv', 'query2_labeled.csv', 'query3_labeled.csv']

with open(FASTA_PATH, 'r') as f:
    sequence_wt = f.readlines()[1].strip()
SEQ_LEN = len(sequence_wt)

df_train = pd.read_csv(TRAIN_PATH)
for qp in QUERY_PATHS:
    if os.path.exists(qp):
        df_train = pd.concat([df_train, pd.read_csv(qp)], ignore_index=True)
df_train = df_train.drop_duplicates(subset='mutant').reset_index(drop=True)

df_test = pd.read_csv(TEST_PATH)

## 2. Feature Engineering (BLOSUM62 + Physicochemical)

In [9]:
blosum62 = substitution_matrices.load("BLOSUM62")

# Kyte-Doolittle Hydrophobicity, Volume, Isoelectric Point
aa_props = {
    'A': [1.8, 88.6, 6.00], 'R': [-4.5, 173.4, 10.76], 'N': [-3.5, 114.1, 5.41],
    'D': [-3.5, 111.1, 2.77], 'C': [2.5, 108.5, 5.07], 'Q': [-3.5, 143.8, 5.65],
    'E': [-3.5, 138.4, 3.22], 'G': [-0.4, 60.1, 5.97], 'H': [-3.2, 153.2, 7.59],
    'I': [4.5, 166.7, 6.02], 'L': [3.8, 166.7, 5.98], 'K': [-3.9, 168.6, 9.74],
    'M': [1.9, 162.9, 5.74], 'F': [2.8, 189.9, 5.48], 'P': [-1.6, 112.7, 6.30],
    'S': [-0.8, 89.0, 5.68], 'T': [-0.7, 116.1, 5.66], 'W': [-0.9, 227.8, 5.89],
    'Y': [-1.3, 193.6, 5.66], 'V': [4.2, 140.0, 5.96]
}

def extract_features(df):
    features = []
    for mut in df['mutant']:
        wt_aa, pos_str, mut_aa = mut[0], mut[1:-1], mut[-1]
        pos = int(pos_str)

        try:
            b_score = blosum62[wt_aa, mut_aa]
        except KeyError:
            b_score = blosum62[mut_aa, wt_aa]

        norm_pos = pos / SEQ_LEN

        wt_p = aa_props.get(wt_aa, [0,0,0])
        mt_p = aa_props.get(mut_aa, [0,0,0])
        hydro_diff = mt_p[0] - wt_p[0]
        vol_diff = mt_p[1] - wt_p[1]
        pi_diff = mt_p[2] - wt_p[2]

        features.append([b_score, norm_pos, hydro_diff, vol_diff, pi_diff, wt_p[0], mt_p[0]])
    return np.array(features)

X_train = extract_features(df_train)
y_train = df_train['DMS_score'].values
X_test  = extract_features(df_test)

## 3. Model Training (XGBoost Ensemble)

In [10]:
# 1. Position-Split CV
train_pos = df_train['mutant'].apply(lambda x: int(x[1:-1])).values
unique_pos = np.unique(train_pos)
np.random.seed(42)
np.random.shuffle(unique_pos)
fold_size = len(unique_pos) // 5
val_spearmans = []

print("Running 5-Fold Position-Split CV...")
for fold in range(5):
    val_pos = set(unique_pos[fold*fold_size:(fold+1)*fold_size])
    val_mask = np.array([p in val_pos for p in train_pos])
    tr_mask = ~val_mask

    model = xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=fold
    )
    model.fit(X_train[tr_mask], y_train[tr_mask])
    y_val_pred = model.predict(X_train[val_mask])
    r, _ = spearmanr(y_train[val_mask], y_val_pred)
    val_spearmans.append(r)
    print(f"  Fold {fold+1} Spearman: {r:.4f}")

print(f"\nHonest CV Spearman: {np.mean(val_spearmans):.4f} ± {np.std(val_spearmans):.4f}")

# 2. Train Final Ensemble on ALL Data
print("\nTraining final XGBoost ensemble ...")
N_MODELS = 5
final_models = []

for i in range(N_MODELS):
    model = xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=i*10
    )
    model.fit(X_train, y_train)
    final_models.append(model)

print("Final ensemble trained successfully!")

Running 5-Fold Position-Split CV...
  Fold 1 Spearman: 0.5674
  Fold 2 Spearman: 0.6348
  Fold 3 Spearman: 0.6850
  Fold 4 Spearman: 0.6134
  Fold 5 Spearman: 0.6555

Honest CV Spearman: 0.6312 ± 0.0397

Training final XGBoost ensemble ...
Final ensemble trained successfully!


## 4. Predictions & Uncertainty

In [ ]:
test_preds = np.array([m.predict(X_test) for m in final_models])
y_pred_mean = np.clip(np.mean(test_preds, axis=0), 0, 1)
y_pred_std  = np.std(test_preds, axis=0)

df_test['DMS_score_predicted'] = y_pred_mean
df_test['uncertainty'] = y_pred_std

## 5. Exports

In [ ]:
# A. Submission
submission = pd.DataFrame({
    'id': range(len(df_test)),
    'DMS_score': df_test['DMS_score_predicted']
})
submission.to_csv('predictions.csv', index=False)
print("Saved predictions.csv")

# B. Top 10 File
top10 = df_test.nlargest(10, 'DMS_score_predicted')['mutant']
with open('top10.txt', 'w') as f:
    for m in top10:
        f.write(f"{m}\n")
print("Saved top10.txt")

# C. Active Learning Query (Upper Confidence Bound Strategy)
train_mutants = set(df_train['mutant'])
df_pool = df_test[~df_test['mutant'].isin(train_mutants)].copy()

# Mixes exploitation (predicted score) with exploration (uncertainty)
kappa = 1.5
df_pool['ucb_score'] = df_pool['DMS_score_predicted'] + (kappa * df_pool['uncertainty'])
query_mutants = df_pool.nlargest(100, 'ucb_score')['mutant']

with open('query.txt', 'w') as f:
    for m in query_mutants:
        f.write(f"{m}\n")
print("Saved query.txt")

Saved predictions_kaggle.csv for Kaggle
Saved top10.txt
Saved query.txt


Kaggle spearman correlation -> 0.41518